# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR\^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
# Show dataset version, release date, and citation
print(f"Version: {metadata.version}")
print(f"Date Published: {metadata.datePublished}")
print(f"Cite As: {getattr(metadata, 'citeAs', 'N/A')}")


## 2. Data Overview
Review available record sets, fields, and their IDs. All dataset entities are referenced by their `@id` values, as per the Croissant schema standard.

In [ ]:
# List all record sets available in the dataset

record_sets = dataset.metadata.recordSet
print(f"Found {len(record_sets)} record set(s):\n")

for i, rs in enumerate(record_sets):
    print(f"[{i}] RecordSet '@id': {rs['@id']}")
    print(f"    Name: {rs.get('name','')}\n    Description: {rs.get('description','')}")
    # Fields in this record set
    fields = rs.get('field', [])
    if fields:
        print("    Fields (by @id):")
        for fld in fields:
            if isinstance(fld, dict):
                print("        -", fld.get('@id', fld))
            else:
                print("        -", fld)
    print()


## 3. Data Extraction
Load data from one or more record sets into DataFrames. Entities (record sets, fields) are referenced by their `@id`. Choose relevant record sets by their `@id` from the overview above.

In [ ]:
# Prepare to extract one or more tables (record sets)

# Example: assuming main record sets are (please adjust if the @id values differ):
main_record_sets = [rs['@id'] for rs in dataset.metadata.recordSet]

dataframes = {}
for rs_id in main_record_sets:
    print(f"Loading records for RecordSet @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records.")
    print(f"Columns: {df.columns.tolist()}\n")
    print(df.head(2))
    print("-----\n")
# Select one main record set for downstream analysis.
if len(main_record_sets):
    chosen_record_set_id = main_record_sets[0]
    print(f"Selected RecordSet '@id': {chosen_record_set_id}")
else:
    chosen_record_set_id = None


## 4. Exploratory Data Analysis (EDA)
We will process data from a record set by referencing field (column) `@id`s. Typical analysis steps include:
- Filtering records using a numeric field.
- Normalizing the values.
- Grouping by categorical field.

> ℹ️ **Tip:** Use the previous code cell output to reference available columns by their `@id`.


In [ ]:
# ---- Update the variables below depending on the schema ----
# Example numeric and group field (adjust for your dataset -- fill from print(df.columns)):

# For demonstration, let us try find numeric and categorical fields:
df = dataframes[chosen_record_set_id]
print("Available columns:", df.columns.tolist())

# Let's pick a likely numeric field (for demonstration):
# Example: 'cr:field/protein_coverage_percent', or similar, based on actual schema
# Please adjust the field name below to match the @id (column) from your data
numeric_field = None
for col in df.columns:
    if ("percent" in str(col).lower() or "abundance" in str(col).lower() or "peptide_count" in str(col).lower()) and df[col].dtype in ['float64', 'int64']:
        numeric_field = col
        break
# If not found, fallback to the first numeric
if not numeric_field:
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
    if len(numeric_cols):
        numeric_field = numeric_cols[0]
print(f"Selected numeric field for filtering: {numeric_field}")

# Example group/categorical field: e.g., cr:field/sample_id
# Find column with 'sample', 'condition', or 'category' in name
group_field = None
for col in df.columns:
    if any(x in str(col).lower() for x in ['sample', 'condition', 'group']):
        group_field = col
        break
print(f"Selected group/categorical field: {group_field}")

# Proceed only if there's a numeric_field
if numeric_field is not None:
    # Remove rows with missing values in this field
    thresh = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > thresh]
    print(f"Filtered {len(filtered_df)} records where {numeric_field} > {thresh:.3f}")

    # Normalize the numeric field (z-score)
    filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

    # Group by 'group_field' if available
    if group_field is not None and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped by {group_field} (showing means):")
        print(grouped_df.head())
else:
    print("No numeric field found for analysis.")


## 5. Visualization
Visualize data distributions and relationships between numeric and categorical fields. Here we create example plots for the selected numeric and group field.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot by group if available
    if group_field is not None and group_field in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field to visualize.")


## 6. Conclusion
We have loaded the FAIR^2 dataset using `mlcroissant`, inspected its schema, and analyzed and visualized its records using `pandas` and `seaborn`.

- All data entities were referenced by their unique `@id` as defined by the Croissant schema.
- We identified numeric and group/categorical fields for filtering, normalization, and grouping.
- Visualizations illustrated the distributions and group differences in the dataset.

Next steps may include feature engineering, advanced modeling, or exporting processed data for downstream analysis.
